# 05 用户分层（RFM 思想）

回答：**谁是好用户？怎么差异化运营？**

RFM 适配：本数据集没有交易金额，用 **购买次数** 替代 Monetary，用 **行为总次数** 作为 Frequency。

In [ ]:
import sys; sys.path.append('..')
from utils.db import query
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

## 1. 计算 RFM 指标

In [ ]:
rfm_sql = """
WITH user_stats AS (
    SELECT 
        user_id,
        COUNT(*) AS frequency,
        MAX(date) AS last_active_date,
        MIN(date) AS first_active_date,
        SUM(CASE WHEN behavior_type='buy' THEN 1 ELSE 0 END) AS monetary,
        SUM(CASE WHEN behavior_type='pv' THEN 1 ELSE 0 END) AS pv_count,
        SUM(CASE WHEN behavior_type='cart' THEN 1 ELSE 0 END) AS cart_count,
        SUM(CASE WHEN behavior_type='fav' THEN 1 ELSE 0 END) AS fav_count
    FROM user_behavior_clean
    WHERE date < '2017-12-08'
    GROUP BY user_id
)
SELECT 
    user_id, frequency, monetary, pv_count, cart_count, fav_count,
    last_active_date, first_active_date,
    julianday('2017-12-07') - julianday(last_active_date) AS recency_days
FROM user_stats
"""
rfm = query(rfm_sql)
print(f'用户数: {len(rfm):,}')
rfm.head()

## 2. RFM 打分（五分位法）

将每个指标分为 1-5 档，5 = 最好（最近活跃、高频、多买）。

In [ ]:
# 用百分位 rank 打分（1-5 档），避免 qcut 的分位点重复问题
def score_rank(series, ascending=True):
    """按百分位 rank 分为 1-5 档，5=最好"""
    pct = series.rank(pct=True, ascending=ascending)
    return pd.cut(pct, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], labels=[1, 2, 3, 4, 5], include_lowest=True).astype(int)

rfm['R_score'] = score_rank(rfm['recency_days'], ascending=False)  # 越小越好
rfm['F_score'] = score_rank(rfm['frequency'], ascending=True)       # 越大越好
rfm['M_score'] = score_rank(rfm['monetary'], ascending=True)        # 越大越好

rfm['RFM_total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print('RFM 得分分布:')
for col in ['R_score', 'F_score', 'M_score']:
    print(f'  {col}: {rfm[col].value_counts().sort_index().to_dict()}')
rfm[['user_id', 'recency_days', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_total']].head(10)

## 3. 用户分层标签

In [ ]:
def segment(row):
    rs, fs, ms = row['R_score'], row['F_score'], row['M_score']
    total = rs + fs + ms
    if total >= 12:
        return '高价值用户'
    elif total >= 9:
        return '活跃用户'
    elif total >= 6:
        return '普通用户'
    elif row['recency_days'] > 7:
        return '流失风险'
    else:
        return '新用户/低频'

rfm['segment'] = rfm.apply(segment, axis=1)

seg_counts = rfm['segment'].value_counts()
seg_pct = (seg_counts / len(rfm) * 100).round(1)
print('用户分层结果:')
print(pd.DataFrame({'用户数': seg_counts, '占比(%)': seg_pct}))

In [ ]:
colors_seg = {'高价值用户': '#C44E52', '活跃用户': '#DD8452', '普通用户': '#55A868', '流失风险': '#4C72B0', '新用户/低频': '#8E44AD'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 饼图
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=[colors_seg[s] for s in seg_counts.index], startangle=90)
axes[0].set_title('用户分层占比')

# 各分层的平均指标
seg_profile = rfm.groupby('segment').agg(
    平均活跃天数=('recency_days', lambda x: (rfm['recency_days'].max() - x).mean()),
    平均行为次数=('frequency', 'mean'),
    平均购买次数=('monetary', 'mean'),
    用户数=('user_id', 'count')
).round(1)
print('各分层画像:')
print(seg_profile)

# 各分层的购买贡献
buy_contribution = rfm.groupby('segment')['monetary'].sum()
axes[1].bar(buy_contribution.index, buy_contribution.values,
            color=[colors_seg[s] for s in buy_contribution.index])
axes[1].set_title('各分层购买次数贡献')
axes[1].set_ylabel('总购买次数')
axes[1].tick_params(axis='x', rotation=20)
total_buys = buy_contribution.sum()
for i, v in enumerate(buy_contribution.values):
    axes[1].text(i, v + 20, f'{v/total_buys*100:.0f}%', ha='center', fontweight='bold')

# 散点图: Recency vs Frequency
for seg in seg_counts.index:
    subset = rfm[rfm['segment'] == seg]
    axes[2].scatter(subset['recency_days'], np.log1p(subset['frequency']),
                    c=colors_seg[seg], label=seg, alpha=0.5, s=10)
axes[2].set_xlabel('距上次活跃天数')
axes[2].set_ylabel('行为次数 (log)')
axes[2].set_title('Recency vs Frequency（分群）')
axes[2].legend(markerscale=3)

plt.tight_layout()
plt.savefig('../data/fig_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 分群行为差异

不同分层用户在行为深度上的差异——这直接指导运营策略。

In [ ]:
seg_behavior = rfm.groupby('segment').agg(
    人均PV=('pv_count', 'mean'),
    人均加购=('cart_count', 'mean'),
    人均收藏=('fav_count', 'mean'),
    人均购买=('monetary', 'mean'),
    加购转化率=('monetary', lambda x: (x > 0).mean() * 100)
).round(1)

print('各分层行为深度:')
print(seg_behavior)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(seg_behavior))
width = 0.2
metrics = ['人均PV', '人均加购', '人均收藏', '人均购买']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar([xi + i*width for xi in x], seg_behavior[metric], width, label=metric, color=color)
ax.set_xticks([xi + 1.5*width for xi in x])
ax.set_xticklabels(seg_behavior.index, rotation=15)
ax.set_ylabel('人均次数')
ax.set_title('各分层行为深度对比')
ax.legend()
plt.tight_layout()
plt.savefig('../data/fig_seg_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 关键发现 + 运营建议

In [ ]:
high_value = rfm[rfm['segment'] == '高价值用户']
at_risk = rfm[rfm['segment'] == '流失风险']
total_buy = rfm['monetary'].sum()
hv_buy = high_value['monetary'].sum()

print('=== 用户分层关键发现 ===')
print(f'1. 高价值用户仅占 {len(high_value)/len(rfm)*100:.1f}%，但贡献了 {hv_buy/total_buy*100:.0f}% 的购买')
print(f'2. 流失风险用户 {len(at_risk):,} 人（{len(at_risk)/len(rfm)*100:.1f}%），需唤醒策略')
print(f'3. 新用户/低频占比最大，需要首次行为引导（push 加购/收藏）')
print(f'4. 加购转化率最高群体：高价值用户')
print()
print('=== 运营建议（按分层）===')
print('高价值用户: VIP 权益 + 新品优先通知 + 会员体系')
print('活跃用户:   推荐加购商品 + 限时折扣，推动首购或复购')
print('普通用户:   发券刺激首次购买 + 引导收藏/加购行为')
print('流失风险:   短信/推送召回 + 大额优惠券')
print('新用户/低频: 新人礼包 + 首单立减 + 行为引导（加购=送积分）')